# 🎯 Audio Prediction with Pretrained Model

This script allows you to perform predictions on new audio files using a pretrained deep learning model. 
It loads the saved model, preprocessing parameters (mean, std, sequence length, number of features), 
and the label encoder, then extracts features from each audio file, normalizes them, and predicts the class. 
The output includes the predicted label and the probabilities for each possible class.


In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import opensmile
import pickle

# 🔹 Audio Paths and Model Files Setup

## 🐶 Breed Name
- `razza` variable contains the breed name (e.g., `"chihuahua"`).
- `dirRazza` capitalizes the first letter for folder naming consistency.

## 📂 Audio File Paths
- `audio_paths` lists all the audio files to process (from test seed).
- Each path points to a `.wav` file for the selected breed.

## 💾 Model and Preprocessing Files
- `model_path`: Path to the saved trained model (`.h5`).
- `preprocess_data_path`: Path to saved preprocessing parameters (`.npz`).
- `encoder_path`: Path to saved `LabelEncoder` (`.pkl`).


In [ ]:
razza = "chihuahua"

dirRazza = razza[0].upper() + razza[1:]

audio_paths = [
    r'D:\Scuola\TESI-2025\Mescalina2017_sorted\Chihuahua\sounds\B-OWN\MAH00055.MP4_50_4.wav',
    r'D:\Scuola\TESI-2025\Mescalina2017_sorted\Chihuahua\sounds\G-NEG\MAH00056.MP4_28.wav',
    r'D:\Scuola\TESI-2025\Mescalina2017_sorted\Chihuahua\sounds\B-ANX\MAH00160.MP4_121.wav',
    r'D:\Scuola\TESI-2025\Mescalina2017_sorted\Chihuahua\sounds\B-NEU\MAH00043.MP4_104_2.wav',
    r'D:\Scuola\TESI-2025\Mescalina2017_sorted\Chihuahua\sounds\B-NEU\00061.MTS_10_1.wav',
    r'D:\Scuola\TESI-2025\Mescalina2017_sorted\Chihuahua\sounds\B-NEU\MAH00043.MP4_45_1.wav',
    r'D:\Scuola\TESI-2025\Mescalina2017_sorted\Chihuahua\sounds\G-NEG\MAH00048.MP4_33_1.wav',
    r'D:\Scuola\TESI-2025\Mescalina2017_sorted\Chihuahua\sounds\W-NEG\MAH00160.MP4_98.wav',
    r'D:\Scuola\TESI-2025\Mescalina2017_sorted\Chihuahua\sounds\B-AGR\00061.MTS_66_2.wav'
]

model_path = f'data/{razza}/best_model_razza.h5'
preprocess_data_path = f'data/{razza}/best_preprocessing_params.npz'
encoder_path = f'data/{razza}/best_label_encoder.pkl'

# 🔹 Load Model and Preprocessing Parameters

## 📦 Load Trained Model
- `model = load_model(model_path)` loads the saved CNN model from disk.

## ⚙️ Load Preprocessing Parameters
- `mean`, `std`: normalization parameters computed from training data.
- `max_len`: maximum sequence length used for padding.
- `n_features`: number of features per audio frame.

## 🏷️ Load Label Encoder
- Loads the saved `LabelEncoder` to decode predicted labels back to class names.


In [11]:
model = load_model(model_path)

data = np.load(preprocess_data_path)
mean = data['mean']
std = data['std']
max_len = int(data['max_len'])
n_features = int(data['n_features'])


with open(encoder_path, 'rb') as f:
    label_encoder = pickle.load(f)


# 🔹 Feature Extraction Utilities

- **`extract_lld_features(audio_path)`**  
  Extracts Low-Level Descriptors (LLDs) from an audio file using the `ComParE_2016` feature set provided by `openSMILE`.  
  Returns a NumPy array of extracted features.

In [4]:
def extract_lld_features(audio_path):
    """Estrae LLD e applica padding se necessario"""
    smile = opensmile.Smile(
        feature_set=opensmile.FeatureSet.ComParE_2016,
        feature_level=opensmile.FeatureLevel.LowLevelDescriptors
    )
    features = smile.process_file(audio_path).values
    return features

# 🔹 Audio Prediction Function

## 🔍 Function Overview
The `predict_audio_file` function takes a single audio file and predicts its class using a trained CNN model.

## 🛠️ Steps Performed
1. **Feature Extraction**  
   Extracts LLD (Low-Level Descriptor) features from the audio file.

2. **Padding / Truncation**  
   - If the sequence is shorter than `max_len`, pads with zeros.  
   - If longer, truncates to `max_len`.

3. **Normalization**  
   Applies standardization using the mean and standard deviation from the training set.

4. **Reshape for CNN Input**  
   Reshapes features to `(1, max_len, n_features, 1)` and optionally downsamples time dimension.

5. **Prediction**  
   - Uses the CNN model to compute probabilities.  
   - Finds the predicted class index and converts it to a label using the `LabelEncoder`.

6. **Return Values**  
   - `pred_label`: predicted class label.  
   - `probs`: dictionary of class probabilities for all possible labels.


In [5]:
def predict_audio_file(audio_path, model, label_encoder, mean, std, max_len, n_features):
    features = extract_lld_features(audio_path)
    
    if features.shape[0] < max_len:
        pad_width = max_len - features.shape[0]
        features = np.pad(features, ((0, pad_width), (0,0)), mode='constant')
    else:
        features = features[:max_len, :]
    
    features = (features - mean) / (std + 1e-8)
    features = features.reshape(1, max_len, n_features, 1)
    features = features[:, ::4, :, :]
    
    preds = model.predict(features)[0]
    
    pred_class_idx = np.argmax(preds)
    pred_label = label_encoder.inverse_transform([pred_class_idx])[0]
    
    probs = {label: float(preds[idx]) for idx, label in enumerate(label_encoder.classes_)}
    
    return pred_label, probs



# 🔹 Batch Audio Prediction and Evaluation

## 🎧 Overview
This code iterates over a list of audio files, predicts their labels using a trained CNN model, and compares predictions with the true labels extracted from the file paths.

## 🔹 Steps Performed

1. **Initialize Counters**  
   - `correct_count`: number of correctly predicted files.  
   - `total_count`: total number of audio files.

2. **Iterate Over Audio Files**  
   For each file in `audio_paths`:
   - Print the file name.
   - Extract the true label from the folder name immediately after "sounds".
   - Use `predict_audio_file` to get the predicted label and class probabilities.

3. **Display Results**  
   - Show the true label and predicted label.
   - Display probabilities for all classes.

4. **Update Accuracy Counter**  
   If the prediction matches the true label, increment `correct_count`.

5. **Final Evaluation**  
   After all files are processed:
   - Print the total number of files.
   - Print the number of correct predictions.
   - Calculate and print overall accuracy as a percentage.


In [ ]:
correct_count = 0
total_count = len(audio_paths)

for audio_path in audio_paths:
    print(f"\n🎧 File: {audio_path}")

    true_label = audio_path.split('\\')[6] 

    predicted_label, probabilities = predict_audio_file(audio_path, model, label_encoder, mean, std, max_len, n_features)

    print(f"True label: {true_label}")
    print(f"Predicted label: {predicted_label}")
    print("Probabilities for each class:")
    for label, prob in probabilities.items():
        print(f"  {label}: {prob:.4f}")

    if predicted_label == true_label:
        correct_count += 1

print(f"\nTotale file: {total_count}")
print(f"Classificazioni corrette: {correct_count}")
print(f"Accuracy: {correct_count / total_count:.2%}")



🎧 File: D:\Scuola\TESI-2025\Mescalina2017_sorted\Chihuahua\sounds\B-OWN\MAH00055.MP4_50_4.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
True label: B-OWN
Predicted label: B-OWN
Probabilities for each class:
  B-AGR: 0.0000
  B-ANX: 0.0006
  B-NEU: 0.0000
  B-OWN: 0.9994
  G-NEG: 0.0000
  W-NEG: 0.0000

🎧 File: D:\Scuola\TESI-2025\Mescalina2017_sorted\Chihuahua\sounds\G-NEG\MAH00056.MP4_28.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
True label: G-NEG
Predicted label: G-NEG
Probabilities for each class:
  B-AGR: 0.0000
  B-ANX: 0.0000
  B-NEU: 0.0000
  B-OWN: 0.0000
  G-NEG: 1.0000
  W-NEG: 0.0000

🎧 File: D:\Scuola\TESI-2025\Mescalina2017_sorted\Chihuahua\sounds\B-ANX\MAH00160.MP4_121.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
True label: B-ANX
Predicted label: B-ANX
Probabilities for each class:
  B-AGR: 0.0000
  B-ANX: 1.0000
  B-NEU: 0.0000
  B-OWN: 0.0000
  G-NEG: 0.0000
  W-NEG: 0.0000

🎧 File: D:\Scuola\TESI-2025\Mescalina2017_sorted\Chihuahua\sounds\B-NEU\MAH00043.MP4_104_2.wav
1/1 ━━━